# Regularization + Scikit-Learn Pipelines

In [1]:
import pandas as pd
import numpy as np
import os
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin

## Import and load data + best trained model

In [2]:
import pickle

with open("../src/feature_lists.pkl", "rb") as f:
    feature_lists = pickle.load(f)

numeric_features = feature_lists["numeric_features"]
categorical_features = feature_lists["categorical_features"]

In [3]:
raw_data_path = Path(f"{os.getcwd()}/../data/application_train.csv")
df = pd.read_csv(raw_data_path)

In [4]:
model_path = Path(f"{os.getcwd()}/../models/best_model.pkl")
with open(model_path, mode='rb') as f:
    best_model = joblib.load(f)

## Define custom transformers to perform preprocessing/feature engineering operations

Here I will define custom pipeline transformers that will perform the operations performed in notebooks 2-3, so that the pipeline will take the training data, fit, and then be able to make predictions on any subsequent testing data that comes through. One thing to be careful of when defining how transformers should `fit()` and `predict()` is data leakage. This is because of the order in which sklearn executes Pipeline operations:
- Firstly, a Pipeline consists of several transformers, either custom or built-in, each with a `fit()` and `predict()` method
- When `pipeline.fit(X_train, y_train)` is called, the transformers learn any parameters, such as median for imputing in `transformer.fit()`
- The transformers then apply any transformations necessary (e.g. imputation) using learned parameters -> The model receives the final processed dataset and trains on it
- It is important that this happens in fit and not transform. Why? Because when we call `predict(X_test)`, the pipeline will apply the transformations to the testing data -> We don't want to impute the testing data with parameters that we learned from the training data; that data is what we used to train the model

### Sentinel replacement

In [5]:
class SentinelTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    
    def transform(self, X: pd.DataFrame):
        # replace days employed sentinel value with NaN
        X_new = X.copy()
        X_new['DAYS_EMPLOYED'] = X_new['DAYS_EMPLOYED'].replace(365243, np.nan)
        return X_new

### Dropping rows from features with more than 40\% missing values

In [6]:
class DropHighMissingValuesTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=40.0):
        self.threshold = threshold

    def fit(self, X: pd.DataFrame, y=None):
        pct_na = pd.DataFrame(X.isna().mean()*100).rename(columns={0:'pct_na'})
    # get columns with more than 40% missing values
        over_40pct_missing = pct_na[pct_na['pct_na'] > 40.0].index
        self.columns_to_drop = [col for col in over_40pct_missing.to_list() if col != 'EXT_SOURCE_1']
        return self

    def transform(self, X: pd.DataFrame):
        X_new = X.copy()
        return X_new.drop(columns=self.columns_to_drop)

### Create missingness indicators

In [7]:
class MissingnessIndicatorTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X: pd.DataFrame, y=None):
        # learn columns that need missingness indicators
        pct_na = pd.DataFrame(X.isna().mean()*100).rename(columns={0:'pct_na'})
        over_5pct_missing_filter = pct_na['pct_na'] > 5.0
        over_5pct_missing = pct_na[over_5pct_missing_filter].index.to_list()
        over_5pct_missing_cols = [col for col in over_5pct_missing if col != 'EXT_SOURCE_1']
        self.missingness_columns = over_5pct_missing_cols
        return self

    def transform(self, X: pd.DataFrame):
        X_new = X.copy()
        for col in self.missingness_columns:
            col_name = f"missingness_{col}"
            # isna returns a boolean series by default, we convert it to int (0/1) so that we can test correlation with TARGET
            X_new[col_name] = X_new[col].isna().astype(int)
        return X_new

### Feature Engineering

In [8]:
class FeatureEngineeringTransfomer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.days_employed_median_ = X['DAYS_EMPLOYED'].median()
        return self
    
    def transform(self, X: pd.DataFrame):
        X_new = X.copy()
        # first re-add age column
        X_new['age'] = abs(X_new['DAYS_BIRTH'] / 365)

        # credit income ratio
        X_new['CREDIT_INCOME_RATIO'] = X_new['AMT_CREDIT'] / X_new['AMT_INCOME_TOTAL']

        # annuity income ratio
        X_new['ANNUITY_INCOME_RATIO'] = X_new['AMT_ANNUITY'] / X_new['AMT_INCOME_TOTAL']

        # credit X_new
        X_new['CREDIT_TERM'] = X_new['AMT_ANNUITY'] / X_new['AMT_CREDIT']

        X_new['DAYS_EMPLOYED'] = X_new['DAYS_EMPLOYED'].replace(np.nan, self.days_employed_median_)
        X_new['DAYS_EMPLOYED_RATIO'] = X_new['DAYS_EMPLOYED'] / X_new['DAYS_BIRTH']

        # income per person
        X_new['INCOME_PER_PERSON'] = X_new['AMT_INCOME_TOTAL'] / X_new['CNT_FAM_MEMBERS']

        # ext source mean
        X_new['EXT_SOURCE_MEAN'] = X_new[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].mean(axis=1)

        # ext source standard deviation
        X_new['EXT_SOURCE_STD'] = X_new[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].std(axis=1)

        # age bucket/bins
        X_new['AGE_BUCKET'] = pd.cut(X_new['age'], bins=[20,30,40,50,60,70,100])

        return X_new

### Engineer Aggregated Features

In [9]:
class AggregatedFeaturesTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.bureau_df_path = Path(f"{os.getcwd()}/../data/bureau.csv")

    def fit(self, X, y=None):

        self.bureau_df_ = pd.read_csv(self.bureau_df_path)

        self.bureau_df_["CREDIT_IS_ACTIVE"] = (
            self.bureau_df_["CREDIT_ACTIVE"] == "Active"
        ).astype(int)

        self.agg_features_ = (
            self.bureau_df_
            .groupby("SK_ID_CURR")
            .agg(
                bureau_records=("SK_ID_BUREAU", "count"),
                num_active_credits=("CREDIT_IS_ACTIVE", "sum"),
                mean_days_credit=("DAYS_CREDIT", "mean"),
                max_credit_sum_overdue=("AMT_CREDIT_SUM_OVERDUE", "max")
            )
            .reset_index()
        )

        self.mean_days_credit_median_ = (
            self.agg_features_["mean_days_credit"].median()
        )

        return self

    def transform(self, X):

        X_new = X.copy()

        X_new = X_new.merge(self.agg_features_, how="left", on="SK_ID_CURR")

        X_new["bureau_records"] = (
            X_new["bureau_records"].fillna(0)
        )

        X_new["num_active_credits"] = (
            X_new["num_active_credits"].fillna(0)
        )

        X_new["max_credit_sum_overdue"] = (
            X_new["max_credit_sum_overdue"].fillna(0)
        )

        X_new["mean_days_credit"] = (
            X_new["mean_days_credit"]
            .fillna(self.mean_days_credit_median_)
        )

        return X_new

### Drop irrelevant columns if they exist

In [10]:
class IrrelevantColumnsTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.columns_to_drop = ['SK_CURR_ID', 'Unnamed: 0']

    def fit(self, X: pd.DataFrame, y=None):
        return self
    
    def transform(self, X: pd.DataFrame):
        X_new = X.copy()
        columns_in_df = [c for c in self.columns_to_drop if c in X.columns]
        return X.drop(columns=columns_in_df)

### Ordinal Encoding Transformer

This transformer encodes the education type column with an ordering

In [11]:
class OrdinalEncodingTransformer(BaseEstimator, TransformerMixin):
    # define hierarchy
    def __init__(self):
        self.mapping = {
            "Lower secondary": 1,
            "Secondary / secondary special": 2,
            "Incomplete higher": 3,
            "Higher education": 4,
            "Academic degree": 5
        }

    def fit(self, X: pd.DataFrame, y=None):
        return self
    
    def transform(self, X: pd.DataFrame):
        X_new = X.copy()
        X_new['NAME_EDUCATION_TYPE'] = X_new['NAME_EDUCATION_TYPE'].map(self.mapping)

        return X_new

### Column Imputing + Encoding

We split the logic based on if the columns are categorical/numerical

In [12]:
numeric_pipeline = Pipeline(
    [("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())]
)

categorical_pipeline = Pipeline(
    [("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))]
)

In [13]:
numeric_features.remove('TARGET')

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

### Defining the pipeline

We chain all of the transformers together, then add the trained best model as the final step

In [14]:
pipeline = Pipeline(
    [
        ('sentinel', SentinelTransformer()),
        ('drop_features', DropHighMissingValuesTransformer(threshold=0.40)),
        ('missingness_indicators', MissingnessIndicatorTransformer()),
        ('feature_engineering', FeatureEngineeringTransfomer()),
        ('aggregated_features', AggregatedFeaturesTransformer()),
        ('irrelevant_features', IrrelevantColumnsTransformer()),
        ('ordinal_encoding', OrdinalEncodingTransformer()),
        ('impute_encode', preprocessor),
        ('model', best_model.named_steps['model'])
    ]
)

In [15]:
# split the data
X, y = df.drop(columns=['TARGET']), df['TARGET']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [16]:
pipeline.fit(X_train, y_train)

,steps,"[('sentinel', ...), ('drop_features', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,threshold,0.4
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False


In [17]:
from sklearn.metrics import roc_auc_score
predictions = pipeline.predict_proba(X_test)
auroc = roc_auc_score(y_test, predictions[:,1])
print(auroc)

0.7535639568599996


### Save pipeline for portability

In [19]:
pipeline_path = Path(f"{os.getcwd()}/../models/credit_scoring_pipeline.pkl")
joblib.dump(pipeline, pipeline_path)

['c:\\Users\\alann\\OneDrive\\Desktop\\Coding\\Jarvis Talent Incubation Training\\jarvis_data_eng_AlanHu\\machine_learning\\module1\\capstone\\notebooks\\..\\models\\credit_scoring_pipeline.pkl']